In [ ]:
import sys
sys.path.append('..')
import os
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from nucml_next.data import NucmlDataset
from nucml_next.baselines import XGBoostEvaluator, DecisionTreeEvaluator

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Data paths (single source of truth)
EXFOR_DATA_PATH = '../data/exfor.parquet'

# Verify EXFOR data exists
if not Path(EXFOR_DATA_PATH).exists():
    raise FileNotFoundError(
        f"EXFOR data not found at {EXFOR_DATA_PATH}\n"
        "Please run: python scripts/ingest_exfor.py "
        "--x4-db data/x4sqlite1.db --output data/exfor_processed.parquet"
    )

print("Imports successful")
print("EXFOR data found")

In [ ]:
# ============================================================================
# USER CONFIGURATION: Feature Tiers and Transformations
# ============================================================================
# Change these settings in ONE place instead of scattered throughout the notebook

from nucml_next.data.selection import TransformationConfig
# ============================================================================
# FEATURE TIER SELECTION
# ============================================================================
# Choose which AME2020/NUBASE2020 enrichment tiers to include
#
# Tier A (13 features) - ALWAYS INCLUDED:
#   - Z, A, N, Energy (nuclear coordinates)
#   - 9-feature Numerical Particle Vector:
#     out_n, out_p, out_a, out_g, out_f, out_t, out_h, out_d, is_met
#
# Tier B (+2 features) - Geometric:
#   + R_fm (nuclear radius)
#   + kR (dimensionless interaction parameter)
#
# Tier C (+7 features) - Energetics: RECOMMENDED FOR BASELINES
#   + Mass_Excess_MeV (mass excess)
#   + Binding_Energy_MeV (total binding energy)
#   + Binding_Per_Nucleon_MeV (B/A)
#   + S_1n_MeV, S_2n_MeV (neutron separation energies)
#   + S_1p_MeV, S_2p_MeV (proton separation energies)
#
# Tier D (+9 features) - Topological:
#   + Spin, Parity (nuclear structure)
#   + Isomer_Level, Half_Life_log10_s (log10-transformed half-life)
#   + Valence_N, Valence_P (distance to magic numbers)
#   + P_Factor (pairing: even-even/odd-odd)
#   + Shell_Closure_N, Shell_Closure_P
#
# Tier E (+8 features) - Complete Q-values:
#   + Q_alpha_MeV, Q_2beta_minus_MeV, Q_ep_MeV, etc.
#   + All 8 reaction Q-values from AME2020

SELECTED_TIERS = ['A', 'B', 'C', 'D', 'E']  # Change tiers HERE (only place to modify)

print(f"Selected Feature Tiers: {SELECTED_TIERS}")
print(f"   Features: Tier A (core + particle vector) + Tier C (energetics)")
print()

# ============================================================================
# TRANSFORMATION CONFIGURATION
# ============================================================================
# Configure log-scaling and feature scaling for ML training.
#
# ORDER OF OPERATIONS (forward transform):
#   1. Log-transform cross-section: sigma' = log10(sigma + epsilon)
#   2. Log-transform energy:        E' = log10(E)
#   3. Scale ALL features:          X' = (X - min) / (max - min)
#
# This ensures the scaler sees compressed log-space values rather than
# raw multi-order-of-magnitude physical values.
#
# For tree-based models (Decision Trees, XGBoost), feature scaling is NOT
# mathematically necessary because trees only use value ordering.
# However, MinMax scaling is cheap and doesn't hurt -- and it prepares
# the pipeline for neural networks where scaling IS required.

TRANSFORMATION_CONFIG = TransformationConfig(
    # ============================================================================
    # Target (cross-section) transformations
    # ============================================================================
    log_target=False,              # Enable log10 transform for cross-sections
                                  # Stabilizes gradients and handles wide range (ub to kb)

    target_epsilon=1e-10,         # Epsilon for log(xs + epsilon) to prevent log(0)
                                  # Increase if you have very small cross-sections

    log_base=10,                  # Logarithm base: 10 | 'e' | 2
                                  # Base-10 is standard in nuclear physics

    # ============================================================================
    # Energy transformations
    # ============================================================================
    log_energy=False,              # Enable log10 transform for energies
                                  # Handles wide energy range (eV to MeV)

    energy_log_base=10,           # Energy log base: 10 | 'e' | 2

    # ============================================================================
    # Feature standardization (MinMax, Z-score, etc.)
    # ============================================================================
    # Order: Log-transforms are applied FIRST, then feature scaling.
    # The scaler is fitted on log-transformed values, so Energy in the
    # scaler's view is log10(E), not raw eV.

    scaler_type='none',         # Feature scaling method:
                                  # 'minmax'   = Min-max scaling to [0,1] [DEFAULT]
                                  # 'standard' = Z-score normalization (X-mu)/sigma
                                  # 'robust'   = Robust scaling using median and IQR
                                  # 'none'     = No scaling
    
    scale_features='all',         # Which columns to scale:
                                  # 'all'  = Scale every numeric column [DEFAULT]
                                  # None   = Auto-detect numeric columns (same as 'all')
                                  # List   = Explicit column names, e.g. ['Z', 'A', 'Energy']
)

print("Transformation Configuration:")
print(TRANSFORMATION_CONFIG)
print()
print("NOTE: MinMax scaling applied to ALL features AFTER log-transforms.")
print("      Trees are scale-invariant, but this prepares the pipeline")
print("      for neural networks and doesn't hurt tree performance.")
print()

# ============================================================================
# UNCERTAINTY WEIGHTING CONFIGURATION
# ============================================================================
# Configure how to use experimental uncertainties during training.
#
# The EXFOR database contains measurement uncertainties for ~66% of cross-section
# values. These uncertainties can be used to weight samples during training,
# giving more influence to precise measurements and less to uncertain ones.
#
# Statistical basis: Inverse-variance weighting (w_i = 1/sigma_i^2) is the
# optimal weighting for least-squares regression when errors are heteroscedastic.

USE_UNCERTAINTY_WEIGHTS = 'xs'    # Uncertainty weighting mode:
                                  # None   = No weighting (equal weight)
                                  # 'xs'   = Weight by cross-section uncertainty (1/sigma_xs^2)
                                  # 'both' = Weight by XS AND energy uncertainty
                                  #          (1/sigma_xs^2 * 1/sigma_E^2)

MISSING_UNCERTAINTY_HANDLING = 'exclude'
                                  # How to handle samples with missing uncertainties
                                  # (only used when USE_UNCERTAINTY_WEIGHTS is not None):
                                  # 'median'  = Assign median weight from valid samples (default)
                                  # 'equal'   = Assign weight of 1.0
                                  # 'exclude' = Exclude samples without valid uncertainty
                                  #             (equivalent to requiring uncertainty)

print("=" * 80)
print("Uncertainty Weighting Configuration:")
print(f"  USE_UNCERTAINTY_WEIGHTS:       {USE_UNCERTAINTY_WEIGHTS}")
print(f"  MISSING_UNCERTAINTY_HANDLING:  '{MISSING_UNCERTAINTY_HANDLING}'")
if USE_UNCERTAINTY_WEIGHTS:
    print(f"\n  NOTE: Uncertainty weighting enabled (mode='{USE_UNCERTAINTY_WEIGHTS}').")
    print("        Samples with lower uncertainty get higher weight.")
    if MISSING_UNCERTAINTY_HANDLING == 'exclude':
        print("\n  NOTE: MISSING_UNCERTAINTY_HANDLING='exclude' will filter to only")
        print("        samples with valid uncertainty (~66% of data).")
print()
print("To change settings, modify SELECTED_TIERS and TRANSFORMATION_CONFIG above")
print("=" * 80)

In [ ]:
# Interactive Outlier Threshold Explorer
# Select any (Z, A, MT) group, adjust the z-score threshold, and inspect
# the smooth mean fit + z-score bands interactively.


# Check for outlier detection columns (supports both local MAD and SVGP methods)
_check_cols = ['z_score', 'experiment_outlier', 'experiment_id', 'point_outlier']
try:
    import pyarrow.parquet as pq
    pq_file = pq.ParquetFile(EXFOR_DATA_PATH)
    _available_cols = pq_file.schema.names
    _cols_to_load = [c for c in _check_cols if c in _available_cols]
    if _cols_to_load:
        _raw_check = pd.read_parquet(EXFOR_DATA_PATH, columns=_cols_to_load)
    else:
        _raw_check = pd.DataFrame()
except Exception:
    _raw_check = pd.DataFrame()

has_outlier_data = 'z_score' in _raw_check.columns and _raw_check['z_score'].notna().any()
has_experiment_outlier = 'experiment_outlier' in _raw_check.columns

if has_outlier_data:
    print("=" * 80)
    print("OUTLIER DETECTION DATA FOUND")
    print("=" * 80)

    if has_experiment_outlier:
        print("Method: Local MAD (smooth mean + energy-local MAD)")
        print("  - point_outlier: individual measurements with z > threshold")
        print("  - experiment_outlier: experiments with high fraction of flagged points")
        print("  - z_score: |deviation from consensus| / local scatter")
        print()
        print("Use 'Color by experiment' toggle to visualize individual experiments")
        print("Discrepant experiments are marked with red X markers")
    else:
        print("Method: Legacy SVGP (point-level only)")
        print("  - z_score: deviation from pooled SVGP fit")

    print("=" * 80)
    print()

    from nucml_next.visualization.threshold_explorer import ThresholdExplorer
    explorer = ThresholdExplorer(EXFOR_DATA_PATH)
    explorer.show()
else:
    print("Outlier columns not found -- run ingestion with --outlier-method to enable")
    print("  Local MAD (recommended):      --outlier-method local_mad")
    print("  Legacy SVGP:                  --outlier-method svgp")
    print("\nProceeding without outlier filtering.\n")

# Store for use in cell 6
_HAS_EXPERIMENT_OUTLIER = has_experiment_outlier
del _raw_check

In [ ]:
# Run after explorer config

# ============================================================================
# GET SETTINGS FROM EXPLORER
# ============================================================================
if has_outlier_data and 'explorer' in dir():
    # Get current settings from explorer checkboxes
    FILTER_SETTINGS = explorer.get_filter_settings()

    print("=" * 80)
    print("OUTLIER FILTER SETTINGS (from explorer)")
    print("=" * 80)
    print(f"  z_threshold:                    {FILTER_SETTINGS['z_threshold']}")
    print(f"  exclude_point_outliers:         {FILTER_SETTINGS['exclude_point_outliers']}")
    print(f"  exclude_discrepant_experiments: {FILTER_SETTINGS['exclude_discrepant_experiments']}")
    print()
    print("These settings will be applied when loading training data in the next cell.")
    print("To change settings, adjust the checkboxes in the explorer and re-run this cell.")
    print("=" * 80)
else:
    # Fallback: no outlier data or explorer not initialized
    FILTER_SETTINGS = {
        'z_threshold': None,
        'exclude_point_outliers': False,
        'exclude_discrepant_experiments': False,
    }
    print("=" * 80)
    print("OUTLIER FILTER SETTINGS")
    print("=" * 80)
    print("No outlier data available - all data will be used for training.")
    print("To enable outlier filtering, run ingestion with --outlier-method local_mad")
    print("=" * 80)

In [ ]:
# ============================================================================
# DATA SELECTION & LOADING
# ============================================================================
# Physics-aware selection with predicate pushdown for efficient loading.

from nucml_next.data import DataSelection
from nucml_next.experiment import HoldoutConfig, ExperimentManager, compute_holdout_metrics

print("=" * 80)
print("DATA SELECTION & LOADING")
print("=" * 80)

# ============================================================================
# PHASE-SPACE HOLDOUT CONFIGURATION
# ============================================================================
# Define holdout rules to measure extrapolation accuracy on unseen data.
# Rules are AND-intersected within, OR-unioned across (any match => holdout).
#
# Supported keys:
#   Z, A            â€” isotope (int)
#   MT              â€” reaction channel (int or list)
#   energy_range    â€” (E_min, E_max) in eV
#   xs_range        â€” (XS_min, XS_max) in barns
#   Entry           â€” EXFOR Entry ID(s) (str or list)
#
# Examples (uncomment to enable):
#   {'Z': 92, 'A': 233}                         â€” hold out ALL U-233 data
#   {'Z': 92, 'A': 235, 'MT': 102,
#    'energy_range': (1e-3, 1.0)}               â€” U-235 capture in resonance region
#   {'MT': 18}                                  â€” hold out ALL fission data
#   {'xs_range': (1e-6, 1e-3)}                  â€” hold out low cross-section points

HOLDOUT_CONFIG = HoldoutConfig(rules=[
    # â”€â”€ Uncomment rules below to enable holdout â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # {'Z': 92, 'A': 233},
    # {'Z': 92, 'A': 235, 'MT': 102, 'energy_range': (1e-3, 1.0)},
    # {'MT': 18},
])

holdout_config = HOLDOUT_CONFIG if HOLDOUT_CONFIG.rules else None

if holdout_config:
    print(f"\nPhase-Space Holdout: {len(holdout_config.rules)} rule(s)")
    print(holdout_config)
else:
    print("\nPhase-Space Holdout: DISABLED")

# ============================================================================
# OUTLIER FILTERING (from explorer settings in cell 6)
# ============================================================================
# These settings come from the ThresholdExplorer checkboxes
_z_threshold = FILTER_SETTINGS.get('z_threshold')
_exclude_point = FILTER_SETTINGS.get('exclude_point_outliers', False)
_exclude_exp = FILTER_SETTINGS.get('exclude_discrepant_experiments', False)

print(f"\nOutlier Filtering:")
print(f"  z_threshold:                    {_z_threshold}")
print(f"  exclude_point_outliers:         {_exclude_point}")
print(f"  exclude_discrepant_experiments: {_exclude_exp}")

# ============================================================================
# DATA SELECTION
# ============================================================================
training_selection = DataSelection(
    # ========================================================================
    # PROJECTILE SELECTION
    # ========================================================================
    projectile='neutron',          # Options: 'neutron' | 'all' | '<code>' | ['<code>', ...]
                                   # 'neutron' = Only neutron-induced reactions (alias for 'n')
                                   # 'all'     = All projectiles, no filtering
                                   # 'n'       = Neutrons only (same as 'neutron')
                                   # 'p'       = Protons only
                                   # 'd'       = Deuterons only
                                   # 'a'       = Alphas only
                                   # 'g'       = Photons (gamma) only
                                   # 't'       = Tritons only
                                   # 'he3'     = Helium-3 only
                                   # ['n','p'] = Neutrons + protons
                                   # ['n','a','d'] = Custom combination

    # ========================================================================
    # ENERGY RANGE (eV)
    # ========================================================================
    energy_min=1e-5,               # Minimum energy in eV (1e-5 = 0.01 eV, thermal neutrons)
    energy_max=2e7,                # Maximum energy in eV (2e7 = 20 MeV, reactor physics upper bound)
                                   # Common ranges:
                                   #   - Thermal: 1e-5 to 1 eV
                                   #   - Resonance: 1 to 1e4 eV
                                   #   - Fast: 1e4 to 2e7 eV (20 MeV)
                                   #   - High energy: up to 1e9 eV (1 GeV)

    # ========================================================================
    # REACTION (MT) MODE SELECTION
    # ========================================================================
    mt_mode='reactor_core',        # Options:
                                   # 'reactor_core'   â†’ Essential for reactor modeling
                                   #                    (MT 2, 4, 16, 18, 102, 103, 107)
                                   #
                                   # 'threshold_only' â†’ Reactions with energy thresholds
                                   #                    (MT 16, 17, 103, 104, 105, 106, 107)
                                   #
                                   # 'fission_details'â†’ Fission breakdown channels
                                   #                    (MT 18, 19, 20, 21, 38)
                                   #
                                   # 'all_physical'   â†’ All MT codes (subject to exclude_mt)
                                   #
                                   # 'custom'         â†’ Use custom_mt_codes list (see below)

    exclude_spectrum_averaged=True, # Exclude MXW, SPA, FIS, AV, BRA, BRS, SDT, FST, TTA
                                    # These are spectrum-averaged (non-monoenergetic) data
                                    # where "Energy" = characteristic spectrum energy,
                                    # not the actual incident energy.
                                    # Set False to include all sf8 types.

    # ========================================================================
    # DATA VALIDITY
    # ========================================================================
    drop_invalid=True,             # Drop NaN or non-positive cross-sections
                                   # Essential for log-transform: log(Ïƒ) requires Ïƒ > 0
                                   # Prevents training instabilities

    # ========================================================================
    # PHASE-SPACE HOLDOUT
    # ========================================================================
    holdout_config=holdout_config, # HoldoutConfig with intersection/union rule logic
                                   # Replaces the legacy holdout_isotopes parameter
                                   # Set to None to disable holdout filtering

    # ========================================================================
    # AME2020/NUBASE2020 ENRICHMENT TIER SELECTION
    # ========================================================================
    tiers=SELECTED_TIERS,          # Using centralized tier configuration from cell 3
    transformation_config=TRANSFORMATION_CONFIG,  # Using centralized transformation config

    # ========================================================================
    # OUTLIER FILTERING (from explorer settings)
    # ========================================================================
    z_threshold=_z_threshold if _exclude_point else None,
    include_outliers=not _exclude_point,
    exclude_discrepant_experiments=_exclude_exp,
)

print("\n" + "-" * 80)
print("Training Selection:")
print(training_selection)

# ============================================================================
# LOAD DATASET
# ============================================================================
print("\n" + "-" * 80)
print("Loading dataset with predicate pushdown...")
dataset_full = NucmlDataset(
    data_path=EXFOR_DATA_PATH,
    mode='tabular',
    selection=training_selection,
)

# Retrieve holdout data (if configured)
df_holdout = dataset_full.get_holdout_data()
if df_holdout is not None:
    print(f"\n[OK] Holdout reserved: {len(df_holdout):,} points "
          f"({df_holdout.groupby(['Z','A']).ngroups} isotopes)")
else:
    print("\n[--] No holdout data")

# ============================================================================
# PROJECT TO TABULAR FORMAT
# ============================================================================
print("\n" + "-" * 80)
print("Projecting to tabular format (particle vector)...")
_extra_meta = ['Energy_Uncertainty'] if USE_UNCERTAINTY_WEIGHTS == 'both' else None
df_tier = dataset_full.to_tabular(extra_metadata=_extra_meta)

print(f"\n[OK] Training set: {len(df_tier):,} samples x {len(df_tier.columns)} columns")
print(f"     Energy range: {df_tier['Energy'].min():.2e} â€“ {df_tier['Energy'].max():.2e} eV")

# ============================================================================
# UNCERTAINTY COVERAGE SUMMARY
# ============================================================================
if 'Uncertainty' in df_tier.columns:
    valid_unc = df_tier['Uncertainty'].notna() & (df_tier['Uncertainty'] > 0)
    pct = 100 * valid_unc.sum() / len(df_tier)
    print(f"     XS uncertainty: {valid_unc.sum():,} / {len(df_tier):,} ({pct:.1f}%)")

# ============================================================================
# FEATURE SUMMARY
# ============================================================================
print("\n" + "-" * 80)
print(f"Feature Tiers: {SELECTED_TIERS}")
TIER_NAMES = {'A': 'Core+Particle', 'B': 'Geometric', 'C': 'Energetics',
              'D': 'Topological', 'E': 'Q-values'}
for t in SELECTED_TIERS:
    print(f"  Tier {t}: {TIER_NAMES.get(t, 'Unknown')}")

# ============================================================================
# DATA DISTRIBUTION
# ============================================================================
print("\n" + "-" * 80)
print("Top 10 Isotopes:")
for (z, a), cnt in dataset_full.df.groupby(['Z', 'A']).size().nlargest(10).items():
    elem = {92:'U', 17:'Cl', 94:'Pu', 26:'Fe', 8:'O', 1:'H', 82:'Pb',
            6:'C', 13:'Al', 7:'N', 11:'Na', 79:'Au'}.get(z, f'Z{z}')
    print(f"  {elem}-{a:3d}: {cnt:>8,}")

print(f"\nTotal: {dataset_full.df.groupby(['Z','A']).ngroups} isotopes, "
      f"{dataset_full.df['MT'].nunique()} MT codes, {len(dataset_full.df):,} points")

df_tier.head()

In [ ]:
# TIER E FIX: load Q-values directly from AME enricher and merge into df_tier
from nucml_next.data.enrichment import AME2020DataEnricher

enricher    = AME2020DataEnricher(data_dir='../data')
enrich_table = enricher.load_all()

# Keep only Z, A and Q-value columns
q_cols_kev = [
    'Q_alpha_keV', 'Q_2beta_minus_keV', 'Q_ep_keV', 'Q_beta_n_keV',
    'Q_4beta_minus_keV', 'Q_d_alpha_keV', 'Q_p_alpha_keV', 'Q_n_alpha_keV',
]
available = [c for c in q_cols_kev if c in enrich_table.columns]
print(f"Available Q-value columns: {available}")

# Ground states only to avoid duplicate rows on merge
merge_table = enrich_table[enrich_table['Isomer_Level'] == 0][['Z', 'A'] + available].copy()

# Convert keV to MeV
q_mev_cols = []
for col in available:
    mev_col = col.replace('_keV', '_MeV')
    merge_table[mev_col] = merge_table[col] / 1000.0
    merge_table = merge_table.drop(columns=[col])
    q_mev_cols.append(mev_col)

# Merge into df_tier
before = len(df_tier)
df_tier = df_tier.merge(merge_table, on=['Z', 'A'], how='left')
assert len(df_tier) == before, "Row count changed after merge"

# Coverage report
print(f"\nMerged {len(q_mev_cols)} Q-value columns into df_tier")
for col in q_mev_cols:
    coverage = df_tier[col].notna().mean() * 100
    flag = "(POOR_COVERAGE)" if coverage < 80 else ""
    print(f"{col} {coverage}%{flag}")

In [ ]:
print(df_tier.columns)

In [ ]:
# MT FILTERING — remove unphysical and leakage MT codes

EXCLUDE_MT = {0, 1}

before = len(df_tier)
df_tier = df_tier[~df_tier["MT"].isin(EXCLUDE_MT)].copy()
after  = len(df_tier)

print(f"Removed MT codes: {EXCLUDE_MT}")
print(f"Rows before:      {before:,}")
print(f"Rows after:       {after:,}")
print(f"Removed:          {before - after:,} ({100*(before-after)/before:.1f}%)")
print(f"\nRemaining MT distribution:")
print(df_tier["MT"].value_counts().sort_index())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from copy import deepcopy
import paper_style

# Helper class for dataset
class XSDataset(Dataset):
    def __init__(self, X, y, w):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
        self.w = torch.from_numpy(w)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]

# Define neural network
class CrossSectionNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),

            nn.Linear(256, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),

            nn.Linear(256, 128),
            nn.GELU(),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# Resonance-upweighted loss function
def physics_loss(preds, targets, log_energies=None,
                 lambda_resonance=5.0,
                 resonance_logE_range=(2.0, 5.0),
                 sample_weights=None):

    # Base weighted loss
    if sample_weights is not None:
        base_loss = (sample_weights * (preds - targets) ** 2).mean()
    else:
        base_loss = nn.functional.mse_loss(preds, targets)

    # Resonance upweighting
    if log_energies is not None:
        lo, hi       = resonance_logE_range
        in_resonance = ((log_energies >= lo) & (log_energies <= hi)).float()
        region_weights = 1.0 + (lambda_resonance - 1.0) * in_resonance
        if sample_weights is not None:
            region_weights = region_weights * sample_weights
        resonance_loss = (region_weights * (preds - targets) ** 2).mean()
    else:
        resonance_loss = base_loss

    return resonance_loss, base_loss.detach()

# Feature preparation for neural network
def prepare_features(df, scaler=None, fit=False):

    # Clean dataset
    df = df.copy()
    df = df[df["CrossSection"] > 0].copy()
    df = df[df["Uncertainty"].notna() & (df["Uncertainty"] > 0)].copy()
    df["logE"] = np.log10(df["Energy"])

    # Drop NaN values
    X = df[FEATURE_COLS].values
    X = np.nan_to_num(X, nan=0.0)

    # Scale features
    if fit:
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    # replace uncertainty weighting with uniform weights (Weigting didn't work)
    w = np.ones(len(df), dtype=np.float32)

    # Set target
    y = np.log10(df["CrossSection"].values).astype(np.float32)

    return X_scaled.astype(np.float32), y, w.astype(np.float32), scaler

FEATURE_COLS = [
    # Tier A — Core nuclear coordinates
    "Z", "A", "N",
    "logE",
    "MT",
    # Tier A — Particle emission vector
    "out_n", "out_p", "out_a", "out_g", "out_f",
    "out_t", "out_h", "out_d", "is_met",
    # Tier B — Geometric
    "R_fm", "kR",
    # Tier C — Energetics
    "Mass_Excess_MeV", "Binding_Energy_MeV", "Binding_Per_Nucleon_MeV",
    "S_1n_MeV", "S_2n_MeV", "S_1p_MeV", "S_2p_MeV",
    # Tier D — Topological
    "Spin", "Parity", "Isomer_Level", "Half_Life_log10_s",
    "Valence_N", "Valence_P", "P_Factor",
    "Shell_Closure_N", "Shell_Closure_P",
    # Tier E — Q-values
    "Q_alpha_MeV", "Q_2beta_minus_MeV", "Q_ep_MeV", "Q_beta_n_MeV",
    "Q_4beta_minus_MeV", "Q_d_alpha_MeV", "Q_p_alpha_MeV", "Q_n_alpha_MeV",
]

In [ ]:
# Set device for neural network
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Train/Test split
df_train, df_val = train_test_split(
    df_tier,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

# Prepare features
X_train, y_train, w_train, scaler = prepare_features(df_train, fit=True)
X_val,   y_val,   w_val,   _      = prepare_features(df_val, scaler=scaler, fit=False)

# Set input dimension
input_dim = X_train.shape[1]

# Create training and validation datasets with the helper class
train_dataset = XSDataset(X_train, y_train, w_train)
val_dataset   = XSDataset(X_val,   y_val,   w_val)

# Load
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4096, shuffle=False)

# Initialise model
nn_model = CrossSectionNet(input_dim).to(device)

# Set resonance weighting parameters
LAMBDA_RESONANCE       = 5.0
RESONANCE_LOGE_RANGE   = (2.0, 5.0) #Epithermal

# Initialise AdamW and scheduler
nn_optimizer = torch.optim.AdamW(nn_model.parameters(), lr=1e-3, weight_decay=1e-4)
nn_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    nn_optimizer, factor=0.5, patience=5
)

print(f"Model initialised on {device}")
print(f"Input dim:      {input_dim}")
print(f"Train batches:  {len(train_loader)}")
print(f"Val batches:    {len(val_loader)}")

In [ ]:
# Training loop params
nn_epochs           = 300
nn_best_state       = None
nn_best_val_loss    = np.inf
nn_patience         = 15
nn_patience_counter = 0

# History tracking
history = {
    "train_loss" : [],
    "val_loss"   : [],
    "lr"         : [],
    "lr_updates" : [],
}
nn_prev_lr = nn_optimizer.param_groups[0]["lr"]

# Loop through epochs
for epoch in range(nn_epochs):

    # Train
    nn_model.train()
    nn_train_loss = 0.0

    # Run optimiser with backpropagation
    for xb, yb, wb in train_loader:
        xb, yb, wb         = xb.to(device), yb.to(device), wb.to(device)
        log_energies_batch = xb[:, 3]

        nn_optimizer.zero_grad()
        preds = nn_model(xb)

        loss, base_l = physics_loss(
            preds, yb,
            log_energies=log_energies_batch,
            lambda_resonance=LAMBDA_RESONANCE,
            resonance_logE_range=RESONANCE_LOGE_RANGE,
            sample_weights=wb,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(nn_model.parameters(), max_norm=1.0)
        nn_optimizer.step()

        nn_train_loss += loss.item() * xb.size(0)

    # Compute training loss
    nn_train_loss /= len(train_loader.dataset)

    # Validation
    nn_model.eval()
    nn_val_loss = 0.0

    with torch.no_grad():
        for xb, yb, wb in val_loader:
            xb, yb, wb         = xb.to(device), yb.to(device), wb.to(device)
            log_energies_batch = xb[:, 3]

            loss, base_l = physics_loss(
                preds=nn_model(xb), targets=yb,
                log_energies=log_energies_batch,
                lambda_resonance=LAMBDA_RESONANCE,
                resonance_logE_range=RESONANCE_LOGE_RANGE,
                sample_weights=wb,
            )
            nn_val_loss += loss.item() * xb.size(0)

    # Compute validation loss
    nn_val_loss /= len(val_loader.dataset)
    nn_scheduler.step(nn_val_loss)

    # Record history
    nn_current_lr = nn_optimizer.param_groups[0]["lr"]
    history["train_loss"].append(nn_train_loss)
    history["val_loss"].append(nn_val_loss)
    history["lr"].append(nn_current_lr)

    if nn_current_lr != nn_prev_lr:
        history["lr_updates"].append(epoch + 1)
        nn_prev_lr = nn_current_lr

    print(f"Epoch {epoch+1:03d} | Train: {nn_train_loss:.4f} | Val: {nn_val_loss:.4f} | LR: {nn_current_lr:.2e}")

    # Early stopping
    if nn_val_loss < nn_best_val_loss:
        nn_best_val_loss    = nn_val_loss
        nn_best_state       = deepcopy(nn_model.state_dict())
        nn_patience_counter = 0
    else:
        nn_patience_counter += 1
        if nn_patience_counter >= nn_patience:
            print("[EARLY STOPPING]")
            break

# Restore best weights
if nn_best_state is not None:
    nn_model.load_state_dict(nn_best_state)

print(f"Training complete — best val loss: {nn_best_val_loss:.4f}")
print(f"LR reductions at epochs: {history['lr_updates']}")

In [ ]:
# Reaction curve prediction

A_target  = 238
Z_target  = 92
MT_target = 102

E_min    = 1e-5
E_max    = 2e7
n_points = 10000

# Energy grid
E_grid    = np.logspace(np.log10(E_min), np.log10(E_max), n_points)
logE_grid = np.log10(E_grid)

# Pull nuclear properties from dataset
base_row = df_tier[
    (df_tier["A"]  == A_target) &
    (df_tier["Z"]  == Z_target) &
    (df_tier["MT"] == MT_target)
]

if len(base_row) == 0:
    raise ValueError(f"Reaction A={A_target}, Z={Z_target}, MT={MT_target} not found in dataset.")

base_row = base_row.iloc[0]

# Build prediction grid from FEATURE_COLS
grid_dict = {}
for col in FEATURE_COLS:
    if col == "logE":
        grid_dict[col] = logE_grid
    else:
        grid_dict[col] = base_row[col]

df_grid = pd.DataFrame(grid_dict)[FEATURE_COLS]
X_grid  = np.nan_to_num(df_grid.values, nan=0.0)
X_grid_scaled = scaler.transform(X_grid)

# NN prediction
nn_model.eval()
with torch.no_grad():
    X_tensor   = torch.tensor(X_grid_scaled, dtype=torch.float32).to(device)
    log_pred   = nn_model(X_tensor).cpu().numpy()

sigma_pred = 10 ** log_pred

# EXFOR true data
df_true = df_tier[
    (df_tier["A"]  == A_target) &
    (df_tier["Z"]  == Z_target) &
    (df_tier["MT"] == MT_target)
].copy()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    df_true["Energy"], df_true["CrossSection"],
    s=8, alpha=0.4, color="steelblue", zorder=2,
    label="EXFOR Data"
)
ax.plot(E_grid, sigma_pred, color="red", linewidth=2.0, label="Neural Net")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy (eV)", fontsize=12)
ax.set_ylabel("Cross Section (barns)", fontsize=12)
ax.set_title(f"Predicted σ(E) — A={A_target}, Z={Z_target}, MT={MT_target}", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, which="both", alpha=0.2)
plt.tight_layout()
plt.show()

# Per-reaction metrics on EXFOR points
df_eval = df_true[df_true["CrossSection"] > 0].copy()
df_eval["logE"]   = np.log10(df_eval["Energy"])
df_eval["log_XS"] = np.log10(df_eval["CrossSection"])

X_eval  = np.nan_to_num(df_eval[FEATURE_COLS].values, nan=0.0)
y_eval  = df_eval["log_XS"].values

X_eval_scaled = scaler.transform(X_eval)
mask_eval     = np.isfinite(X_eval_scaled).all(axis=1) & np.isfinite(y_eval)
X_eval_scaled = X_eval_scaled[mask_eval]
y_eval        = y_eval[mask_eval]

nn_model.eval()
with torch.no_grad():
    nn_eval_pred = nn_model(
        torch.tensor(X_eval_scaled, dtype=torch.float32).to(device)
    ).cpu().numpy()

rmse = np.sqrt(mean_squared_error(y_eval, nn_eval_pred))
r2   = r2_score(y_eval, nn_eval_pred)
print(f"\nPer-reaction metrics on EXFOR points:")
print(f"Log RMSE: {rmse:.4f}  (avg error ~{10**rmse:.2f}x in cross section)")
print(f"Log R²:   {r2:.4f}")

In [ ]:
# TREE MODEL SETUP

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import numpy as np

TREE_FEATURE_COLS = [
    # Tier A — Core nuclear coordinates
    "Z", "A", "N",
    "logE",
    "MT",
    # Tier A — Particle emission vector
    "out_n", "out_p", "out_a", "out_g", "out_f",
    "out_t", "out_h", "out_d", "is_met",
]

# Set constants
TREE_SUBSAMPLE_FRAC = 0.05
TREE_CV_FOLDS       = 2
RANDOM_STATE        = 42

# Clean dataset
df_trees = df_tier.copy()
df_trees = df_trees[df_trees["CrossSection"] > 0].copy()
df_trees["logE"]   = np.log10(df_trees["Energy"])
df_trees["log_XS"] = np.log10(df_trees["CrossSection"])

# Uniform weights again, unfortunately
tree_weights = np.ones(len(df_trees), dtype=np.float32)

# Create features and target
X_trees = df_trees[TREE_FEATURE_COLS].fillna(0).values
y_trees = df_trees["log_XS"].values

mask = np.isfinite(X_trees).all(axis=1) & np.isfinite(y_trees)
X_trees      = X_trees[mask]
y_trees      = y_trees[mask]
tree_weights = tree_weights[mask]

# Train/test split
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_trees, y_trees, tree_weights,
    test_size=0.1, random_state=RANDOM_STATE
)

# Subsample for hyperparam search
n_sub = int(len(X_tr) * TREE_SUBSAMPLE_FRAC)
rng   = np.random.default_rng(RANDOM_STATE)
idx   = rng.choice(len(X_tr), size=n_sub, replace=False)
X_sub, y_sub, w_sub = X_tr[idx], y_tr[idx], w_tr[idx]

print(f"Full train:    {len(X_tr):,} samples")
print(f"Search subset: {len(X_sub):,} samples ({TREE_SUBSAMPLE_FRAC*100:.0f}%)")
print(f"Test:          {len(X_te):,} samples")
print(f"Tree features: {len(TREE_FEATURE_COLS)}")

In [ ]:
# Eval helper

def evaluate_tree(name, model, X_te, y_te, w_te=None):
    y_pred   = model.predict(X_te)
    log_rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    log_r2   = r2_score(y_te, y_pred)

    # Weighted log RMSE
    if w_te is not None:
        w_norm       = w_te / w_te.mean()
        weighted_mse = np.average((y_te - y_pred) ** 2, weights=w_norm)
        w_log_rmse   = np.sqrt(weighted_mse)
    else:
        w_log_rmse = None

    sigma_pred = 10 ** y_pred
    sigma_true = 10 ** y_te
    lin_rmse   = np.sqrt(mean_squared_error(sigma_true, sigma_pred))

    print(f"  {name}")
    print("")
    print(f"Log RMSE (unweighted):  {log_rmse:.4f}")
    if w_log_rmse:
        print(f"Log RMSE (weighted):    {w_log_rmse:.4f}")
    print(f"Log R²:                 {log_r2:.4f}")
    print(f"Linear RMSE (barns):    {lin_rmse:.4e}")

    return {
        "model":        name,
        "log_rmse":     log_rmse,
        "w_log_rmse":   w_log_rmse,
        "log_r2":       log_r2,
        "lin_rmse":     lin_rmse,
    }

In [ ]:
# Decision Tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# Define objective function
def dt_objective(params):
    model = DecisionTreeRegressor(
        max_depth=int(params["max_depth"]),
        min_samples_split=int(params["min_samples_split"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(
        model, X_sub, y_sub,
        cv=TREE_CV_FOLDS,
        scoring="neg_mean_squared_error",
        params={"sample_weight": w_sub},
        n_jobs=-1,
    )
    return {"loss": -scores.mean(), "status": STATUS_OK}

# Search space
dt_space = {
    "max_depth":          hp.quniform("max_depth", 10, 40, 2),
    "min_samples_split":  hp.quniform("min_samples_split", 2, 20, 1),
    "min_samples_leaf":   hp.quniform("min_samples_leaf", 1, 20, 1),
}

# Run hyperopt search
dt_trials = Trials()
best_dt   = fmin(
    fn=dt_objective, space=dt_space, algo=tpe.suggest,
    max_evals=40, trials=dt_trials,
    rstate=np.random.default_rng(RANDOM_STATE),
)

# Retrain on full data with sample weights
dt_model = DecisionTreeRegressor(
    max_depth=int(best_dt["max_depth"]),
    min_samples_split=int(best_dt["min_samples_split"]),
    min_samples_leaf=int(best_dt["min_samples_leaf"]),
    random_state=RANDOM_STATE,
)
dt_model.fit(X_tr, y_tr, sample_weight=w_tr)
dt_metrics = evaluate_tree("Decision Tree", dt_model, X_te, y_te, w_te)
print(f"\n  Best params: {best_dt}")

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor

# Define objective function
def rf_objective(params):
    model = RandomForestRegressor(
        n_estimators=int(params["n_estimators"]),
        max_depth=int(params["max_depth"]),
        min_samples_split=int(params["min_samples_split"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_features=params["max_features"],
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(
        model, X_sub, y_sub,
        cv=TREE_CV_FOLDS,
        scoring="neg_mean_squared_error",
        params={"sample_weight": w_sub},
        n_jobs=-1,
    )
    return {"loss": -scores.mean(), "status": STATUS_OK}

# Search space
rf_space = {
    "n_estimators":      hp.quniform("n_estimators", 100, 500, 50),
    "max_depth":         hp.quniform("max_depth", 10, 40, 2),
    "min_samples_split": hp.quniform("min_samples_split", 2, 20, 1),
    "min_samples_leaf":  hp.quniform("min_samples_leaf", 1, 10, 1),
    "max_features":      hp.choice("max_features", [0.5, 0.7, 1.0, "sqrt"]),
}

# Run hyperopt search
rf_trials = Trials()
best_rf   = fmin(
    fn=rf_objective, space=rf_space, algo=tpe.suggest,
    max_evals=40, trials=rf_trials,
    rstate=np.random.default_rng(RANDOM_STATE),
)
max_features_choices = [0.5, 0.7, 1.0, "sqrt"]

# Retrain on best feature set
rf_model = RandomForestRegressor(
    n_estimators=min(int(best_rf["n_estimators"]), 100),
    max_depth=min(int(best_rf["max_depth"]), 20),
    min_samples_split=int(best_rf["min_samples_split"]),
    min_samples_leaf=int(best_rf["min_samples_leaf"]),
    max_features=max_features_choices[best_rf["max_features"]],
    n_jobs=1,
    random_state=RANDOM_STATE,
)

n_rf   = int(len(X_tr) * 0.3)
idx_rf = np.random.default_rng(RANDOM_STATE).choice(len(X_tr), size=n_rf, replace=False)

rf_model.fit(X_tr[idx_rf], y_tr[idx_rf], sample_weight=w_tr[idx_rf])
rf_metrics = evaluate_tree("Random Forest", rf_model, X_te, y_te, w_te)
print(f"\n  Best params: {best_rf}")

In [ ]:
# XGBoost
from xgboost import XGBRegressor

# Train/test split for XGBoost separately
X_xgb_tr, X_xgb_val, y_xgb_tr, y_xgb_val, w_xgb_tr, w_xgb_val = train_test_split(
    X_tr, y_tr, w_tr,
    test_size=0.1, random_state=RANDOM_STATE
)

# Define objective function
def xgb_objective(params):
    model = XGBRegressor(
        n_estimators=int(params["n_estimators"]),
        max_depth=int(params["max_depth"]),
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=int(params["min_child_weight"]),
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        objective="reg:squarederror",
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbosity=0,
    )
    scores = cross_val_score(
        model, X_sub, y_sub,
        cv=TREE_CV_FOLDS,
        scoring="neg_mean_squared_error",
        params={"sample_weight": w_sub},
        n_jobs=-1,
    )
    return {"loss": -scores.mean(), "status": STATUS_OK}

# Search space
xgb_space = {
    "n_estimators":     hp.quniform("n_estimators", 1000, 10000, 1000),
    "max_depth":        hp.quniform("max_depth", 4, 12, 1),
    "learning_rate":    hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
    "subsample":        hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
    "min_child_weight": hp.quniform("min_child_weight", 1, 20, 1),
    "reg_alpha":        hp.loguniform("reg_alpha",  np.log(1e-4), np.log(10)),
    "reg_lambda":       hp.loguniform("reg_lambda", np.log(1e-4), np.log(10)),
}

# Run hyperopt search
xgb_trials = Trials()
best_xgb   = fmin(
    fn=xgb_objective, space=xgb_space, algo=tpe.suggest,
    max_evals=40, trials=xgb_trials,
    rstate=np.random.default_rng(RANDOM_STATE),
)

# Retrain on best feature set
xgb_model = XGBRegressor(
    n_estimators=int(best_xgb["n_estimators"]),
    max_depth=int(best_xgb["max_depth"]),
    learning_rate=best_xgb["learning_rate"],
    subsample=best_xgb["subsample"],
    colsample_bytree=best_xgb["colsample_bytree"],
    min_child_weight=int(best_xgb["min_child_weight"]),
    reg_alpha=best_xgb["reg_alpha"],
    reg_lambda=best_xgb["reg_lambda"],
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=0,
    early_stopping_rounds=20,
)
xgb_model.fit(
    X_xgb_tr, y_xgb_tr,
    sample_weight=w_xgb_tr,
    eval_set=[(X_xgb_val, y_xgb_val)],
    verbose=False,
)
xgb_metrics = evaluate_tree("XGBoost", xgb_model, X_te, y_te, w_te)

print(f"\nBest n_estimators (early stopped): {xgb_model.best_iteration}")
print(f"Best params:")
print(f"n_estimators:    {int(best_xgb['n_estimators'])}")
print(f"max_depth:       {int(best_xgb['max_depth'])}")
print(f"learning_rate:   {best_xgb['learning_rate']:.4f}")
print(f"subsample:       {best_xgb['subsample']:.4f}")
print(f"colsample_bytree:{best_xgb['colsample_bytree']:.4f}")
print(f"min_child_weight:{int(best_xgb['min_child_weight'])}")
print(f"reg_alpha (L1):  {best_xgb['reg_alpha']:.6f}")
print(f"reg_lambda (L2): {best_xgb['reg_lambda']:.6f}")

In [ ]:
# Model Comparison: Numerical

# NN metrics
def get_nn_preds(loader, model, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb, wb in loader:
            xb = xb.to(device)
            preds.append(model(xb).cpu().numpy())
            trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

nn_preds_log, nn_true_log = get_nn_preds(val_loader, nn_model, device)

def compute_metrics(name, y_true, y_pred):
    log_rmse    = np.sqrt(np.mean((y_pred - y_true) ** 2))
    log_r2      = r2_score(y_true, y_pred)
    linear_rmse = np.sqrt(np.mean((10**y_pred - 10**y_true) ** 2))
    return {
        "Model":            name,
        "Log RMSE":         log_rmse,
        "Log R²":           log_r2,
        "Linear RMSE (b)":  linear_rmse,
    }

# Loop through tree models to get metrics
rows = [
    compute_metrics("Decision Tree", y_te, dt_model.predict(X_te)),
    compute_metrics("Random Forest",  y_te, rf_model.predict(X_te)),
    compute_metrics("XGBoost",        y_te, xgb_model.predict(X_te)),
    compute_metrics("Neural Network", nn_true_log, nn_preds_log),
]

metrics_df = pd.DataFrame(rows).set_index("Model")

print("ALL MODEL COMPARISON")
print("")
print(metrics_df.to_string(float_format="{:.4f}".format))

In [ ]:
# Plot hyperparam search convergence

fig, axes = paper_style.fig("double", ncols=3, sharey=True)

search_results = [
    ("Decision Tree",  dt_trials),
    ("Random Forest",  rf_trials),
    ("XGBoost",        xgb_trials),
]

for ax, (name, trials) in zip(axes, search_results):
    losses      = [t["result"]["loss"] for t in trials.trials]
    best_so_far = np.minimum.accumulate(losses)
    trial_nums  = np.arange(1, len(losses) + 1)

    ax.scatter(trial_nums, losses,
               alpha=0.4, s=8,
               color=paper_style.COLORS["sky"],
               label="Trial loss")
    ax.plot(trial_nums, best_so_far,
            color=paper_style.COLORS["red"],
            linewidth=1.5,
            label="Best so far")

    ax.set_title(name)
    ax.set_xlabel("Trial")
    if ax == axes[0]:
        ax.set_ylabel("Loss (MSE)")
    ax.legend()

    ax.annotate(
        f"Best: {best_so_far[-1]:.4f}",
        xy=(len(losses), best_so_far[-1]),
        xytext=(-40, 10),
        textcoords="offset points",
        fontsize=7,
        arrowprops=dict(arrowstyle="->",
                        color=paper_style.COLORS["red"],
                        lw=0.8),
    )

plt.tight_layout()
paper_style.save("hyperparameter_convergence.pdf")
plt.show()

In [ ]:
# Feature Importances

fig, axes = paper_style.fig("double", ncols=3, sharey=True)

for ax, (name, importances) in zip(axes, [
    ("Decision Tree", dt_model.feature_importances_),
    ("Random Forest",  rf_model.feature_importances_),
    ("XGBoost",        xgb_model.feature_importances_),
]):
    order = np.argsort(importances)
    ax.barh(
        [TREE_FEATURE_COLS[i] for i in order],
        importances[order],
        color=paper_style.COLORS["blue"],
        edgecolor="none",
        height=0.7,
    )
    ax.set_title(name)
    ax.set_xlabel("Importance")
    if ax != axes[0]:
        ax.tick_params(labelleft=False)

plt.tight_layout()
paper_style.save("feature_importance.pdf")
plt.show()

In [ ]:
# Tree predictions
A_target  = 235
Z_target  = 92
MT_target = 18

E_min    = 1e-5
E_max    = 2e8
n_points = 10000

E_grid    = np.logspace(np.log10(E_min), np.log10(E_max), n_points)
logE_grid = np.log10(E_grid)

base_row = df_tier[
    (df_tier["A"]  == A_target) &
    (df_tier["Z"]  == Z_target) &
    (df_tier["MT"] == MT_target)
]
if len(base_row) == 0:
    raise ValueError(f"Reaction A={A_target}, Z={Z_target}, MT={MT_target} not found.")
base_row = base_row.iloc[0]

grid_dict = {"logE": logE_grid}
for col in TREE_FEATURE_COLS:
    if col != "logE":
        grid_dict[col] = base_row[col]
df_grid = pd.DataFrame(grid_dict)[TREE_FEATURE_COLS]
X_grid  = np.nan_to_num(df_grid.values, nan=0.0)

dt_log_pred  = dt_model.predict(X_grid)
rf_log_pred  = rf_model.predict(X_grid)
xgb_log_pred = xgb_model.predict(X_grid)

rf_tree_preds = np.stack(
    [tree.predict(X_grid) for tree in rf_model.estimators_], axis=0
)
rf_std  = rf_tree_preds.std(axis=0)
rf_lo   = 10 ** (rf_log_pred - rf_std)
rf_hi   = 10 ** (rf_log_pred + rf_std)

sigma_dt  = 10 ** dt_log_pred
sigma_rf  = 10 ** rf_log_pred
sigma_xgb = 10 ** xgb_log_pred

df_true = df_tier[
    (df_tier["A"]  == A_target) &
    (df_tier["Z"]  == Z_target) &
    (df_tier["MT"] == MT_target)
].copy()

fig, ax = paper_style.fig("wide")

ax.scatter(
    df_true["Energy"], df_true["CrossSection"],
    s=4, alpha=0.3,
    color=paper_style.COLORS["sky"],
    zorder=2, label="EXFOR Data",
    rasterized=True,
)
ax.plot(E_grid, sigma_dt,  linewidth=1.2, linestyle="--",
        color=paper_style.COLORS["orange"],  label="Decision Tree")
ax.plot(E_grid, sigma_rf,  linewidth=1.2, linestyle="-.",
        color=paper_style.COLORS["green"],   label="Random Forest")
ax.fill_between(E_grid, rf_lo, rf_hi,
                alpha=0.15,
                color=paper_style.COLORS["green"],
                label=r"RF $\pm1\sigma$ (tree variance)")
ax.plot(E_grid, sigma_xgb, linewidth=1.2, linestyle=":",
        color=paper_style.COLORS["purple"],  label="XGBoost")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"$\sigma(E)$ (barns)")
ax.legend(ncol=2)

plt.tight_layout()
paper_style.save("comparison_trees.pdf")
plt.show()

In [ ]:
# Load evaluated data for comparison
import ND_parser

u235_df       = ND_parser.read_END_CSV("u235_fission_eval.csv").drop(index=[0, 1])
u235_df["Z"]  = 92
u235_df["A"]  = 235
u235_df["MT"] = 18
co59_df       = ND_parser.read_END_CSV("co59_capture_eval.csv").drop(index=[0, 1])
co59_df["Z"]  = 27
co59_df["A"]  = 59
co59_df["MT"] = 102

u235_df.head()

In [ ]:
# Per-reaction Metrics on validation set

from scipy.interpolate import interp1d

LIBRARY_COLS = ["TENDL-2019", "JENDL-4.0u", "JEFF-4.0",
                "ENDF/B-VIII.0", "CENDL-3.2", "BROND-3.1"]

REACTIONS = [
    {"name": "U-235 (n,f)",     "lib_df": u235_df,  "A": 235, "Z": 92, "MT": 18},
    {"name": "Co-59 (n,gamma)", "lib_df": co59_df,  "A": 59,  "Z": 27, "MT": 102},
]

for rxn in REACTIONS:
    A, Z, MT = rxn["A"], rxn["Z"], rxn["MT"]
    lib_df   = rxn["lib_df"].copy().sort_values("Incident energy")

    print(f"  {rxn['name']}")
    print(f"")

    # Get EXFOR points for this reaction (val set only)
    df_exfor_all = df_tier[
        (df_tier["A"]  == A) &
        (df_tier["Z"]  == Z) &
        (df_tier["MT"] == MT) &
        (df_tier["CrossSection"] > 0)
    ].copy()

    in_val   = df_exfor_all.index.isin(df_val.index)
    df_exfor = df_exfor_all[in_val].copy()

    print(f"Total EXFOR points:  {len(df_exfor_all):,}")
    print(f"Val set points only: {len(df_exfor):,}")

    if len(df_exfor) < 5:
        print(f"Too few val points to compare: skipping")
        continue

    df_exfor["logE"]   = np.log10(df_exfor["Energy"])
    df_exfor["log_XS"] = np.log10(df_exfor["CrossSection"])
    y_exfor    = df_exfor["log_XS"].values
    logE_exfor = df_exfor["logE"].values

    # Build prediction grids at val EXFOR energies
    base_row = df_tier[
        (df_tier["A"] == A) & (df_tier["Z"] == Z) & (df_tier["MT"] == MT)
    ].iloc[0]

    grid_tree = {"logE": logE_exfor}
    for col in TREE_FEATURE_COLS:
        if col != "logE":
            grid_tree[col] = base_row[col]
    X_tree = np.nan_to_num(
        pd.DataFrame(grid_tree)[TREE_FEATURE_COLS].values, nan=0.0
    )

    grid_nn = {"logE": logE_exfor}
    for col in FEATURE_COLS:
        if col != "logE":
            grid_nn[col] = base_row[col]
    X_nn = scaler.transform(np.nan_to_num(
        pd.DataFrame(grid_nn)[FEATURE_COLS].values, nan=0.0
    ))

    nn_model.eval()
    with torch.no_grad():
        nn_preds = nn_model(
            torch.tensor(X_nn, dtype=torch.float32).to(device)
        ).cpu().numpy()

    model_predictions = {
        "Decision Tree": (dt_model.predict(X_tree), y_exfor),
        "Random Forest":  (rf_model.predict(X_tree), y_exfor),
        "XGBoost":        (xgb_model.predict(X_tree), y_exfor),
        "Neural Network": (nn_preds,                  y_exfor),
    }

    # Interpolate each library onto val EXFOR energies
    for lib_name in LIBRARY_COLS:
        valid = lib_df[lib_name] > 0
        if valid.sum() < 5:
            continue

        lib_energies = lib_df.loc[valid, "Incident energy"].values
        lib_log_xs   = np.log10(lib_df.loc[valid, lib_name].values)

        in_range = (
            (df_exfor["Energy"].values >= lib_energies.min()) &
            (df_exfor["Energy"].values <= lib_energies.max())
        )
        if in_range.sum() < 5:
            continue

        interpolator = interp1d(
            np.log10(lib_energies), lib_log_xs,
            kind="linear", bounds_error=False, fill_value=np.nan
        )
        lib_interp = interpolator(logE_exfor[in_range])
        finite     = np.isfinite(lib_interp)

        model_predictions[lib_name] = (
            lib_interp[finite],
            y_exfor[in_range][finite],
        )

    # Print results table
    print(f"\nModel/Library Log RMSE Log R² N")
    print("")

    for name, (preds, y_compare) in model_predictions.items():
        rmse = np.sqrt(np.mean((preds - y_compare) ** 2))
        r2   = r2_score(y_compare, preds)
        print(f"  {name} {rmse} {r2} {len(y_compare)}")

In [ ]:
# Full Visual Model Comparison

# Build separate grids for tree and NN feature sets
grid_dict_tree = {"logE": logE_grid}
for col in TREE_FEATURE_COLS:
    if col != "logE":
        grid_dict_tree[col] = base_row[col]
X_grid_tree = np.nan_to_num(
    pd.DataFrame(grid_dict_tree)[TREE_FEATURE_COLS].values, nan=0.0
)

grid_dict_nn = {"logE": logE_grid}
for col in FEATURE_COLS:
    if col != "logE":
        grid_dict_nn[col] = base_row[col]
X_grid_nn = np.nan_to_num(
    pd.DataFrame(grid_dict_nn)[FEATURE_COLS].values, nan=0.0
)
X_grid_nn_scaled = scaler.transform(X_grid_nn)

# Tree predictions
dt_log_pred  = dt_model.predict(X_grid_tree)
rf_log_pred  = rf_model.predict(X_grid_tree)
xgb_log_pred = xgb_model.predict(X_grid_tree)

rf_tree_preds = np.stack(
    [tree.predict(X_grid_tree) for tree in rf_model.estimators_], axis=0
)
rf_std = rf_tree_preds.std(axis=0)
rf_lo  = 10 ** (rf_log_pred - rf_std)
rf_hi  = 10 ** (rf_log_pred + rf_std)

sigma_dt  = 10 ** dt_log_pred
sigma_rf  = 10 ** rf_log_pred
sigma_xgb = 10 ** xgb_log_pred

# NN prediction
nn_model.eval()
with torch.no_grad():
    nn_log_pred = nn_model(
        torch.tensor(X_grid_nn_scaled, dtype=torch.float32).to(device)
    ).cpu().numpy()

sigma_nn = 10 ** nn_log_pred

# Plot
fig, ax = paper_style.fig("wide")

ax.scatter(
    df_true["Energy"], df_true["CrossSection"],
    s=4, alpha=0.3,
    color=paper_style.COLORS["sky"],
    zorder=2, label="EXFOR Data",
    rasterized=True,
)
ax.plot(E_grid, sigma_dt,  linewidth=1.2, linestyle="--",
        color=paper_style.COLORS["orange"],  label="Decision Tree")
ax.plot(E_grid, sigma_rf,  linewidth=1.2, linestyle="-.",
        color=paper_style.COLORS["green"],   label="Random Forest")
ax.fill_between(E_grid, rf_lo, rf_hi,
                alpha=0.15,
                color=paper_style.COLORS["green"],
                label=r"RF $\pm1\sigma$")
ax.plot(E_grid, sigma_xgb, linewidth=1.2, linestyle=":",
        color=paper_style.COLORS["purple"],  label="XGBoost")
ax.plot(E_grid, sigma_nn,  linewidth=1.8, linestyle="-",
        color=paper_style.COLORS["red"],     label="Neural Network")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"$\sigma(E)$ (barns)")
ax.legend(ncol=2)

plt.tight_layout()
paper_style.save("full_model_comparison.pdf")
plt.show()

# Per-reaction metrics on EXFOR points
df_eval = df_true[df_true["CrossSection"] > 0].copy()
df_eval["logE"]   = np.log10(df_eval["Energy"])
df_eval["log_XS"] = np.log10(df_eval["CrossSection"])
y_eval            = df_eval["log_XS"].values

# Tree eval grid
X_eval_tree = np.nan_to_num(df_eval[TREE_FEATURE_COLS].values, nan=0.0)
mask_tree   = np.isfinite(X_eval_tree).all(axis=1) & np.isfinite(y_eval)

# NN eval grid
X_eval_nn        = np.nan_to_num(df_eval[FEATURE_COLS].values, nan=0.0)
X_eval_nn_scaled = scaler.transform(X_eval_nn)
mask_nn          = np.isfinite(X_eval_nn_scaled).all(axis=1) & np.isfinite(y_eval)

nn_model.eval()
with torch.no_grad():
    nn_eval_pred = nn_model(
        torch.tensor(X_eval_nn_scaled[mask_nn], dtype=torch.float32).to(device)
    ).cpu().numpy()

# Display per-reaction metrics
print(f"\nPer-reaction metrics on EXFOR points:")
print(f"Model Log RMSE Log R² N points")
print("")

for name, preds, mask in [
    ("Decision Tree",  dt_model.predict(X_eval_tree[mask_tree]), mask_tree),
    ("Random Forest",  rf_model.predict(X_eval_tree[mask_tree]), mask_tree),
    ("XGBoost",        xgb_model.predict(X_eval_tree[mask_tree]), mask_tree),
    ("Neural Network", nn_eval_pred,                              mask_nn),
]:
    y = y_eval[mask]
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2   = r2_score(y, preds)
    print(f"{name} {rmse} {r2} {mask.sum()}")

In [ ]:
# Full Model comparison with ENDFB
A_target  = 59
Z_target  = 27
MT_target = 102

lib_df    = co59_df
lib_label = "ENDF/B-VIII.0"

E_min    = 1e-5
E_max    = 2e7
n_points = 10000

# Energy grid
E_grid    = np.logspace(np.log10(E_min), np.log10(E_max), n_points)
logE_grid = np.log10(E_grid)

# Nuclear properties from df_tier
base_row = df_tier[
    (df_tier["A"]  == A_target) &
    (df_tier["Z"]  == Z_target) &
    (df_tier["MT"] == MT_target)
].iloc[0]

df_true = df_tier[
    (df_tier["A"]  == A_target) &
    (df_tier["Z"]  == Z_target) &
    (df_tier["MT"] == MT_target) &
    (df_tier["CrossSection"] > 0)
].copy()

# Build prediction grids
grid_dict_tree = {"logE": logE_grid}
for col in TREE_FEATURE_COLS:
    if col != "logE":
        grid_dict_tree[col] = base_row[col]
X_grid_tree = np.nan_to_num(
    pd.DataFrame(grid_dict_tree)[TREE_FEATURE_COLS].values, nan=0.0
)

grid_dict_nn = {"logE": logE_grid}
for col in FEATURE_COLS:
    if col != "logE":
        grid_dict_nn[col] = base_row[col]
X_grid_nn_scaled = scaler.transform(np.nan_to_num(
    pd.DataFrame(grid_dict_nn)[FEATURE_COLS].values, nan=0.0
))

# Model predictions
dt_log_pred  = dt_model.predict(X_grid_tree)
rf_log_pred  = rf_model.predict(X_grid_tree)
xgb_log_pred = xgb_model.predict(X_grid_tree)

rf_tree_preds = np.stack(
    [tree.predict(X_grid_tree) for tree in rf_model.estimators_], axis=0
)
rf_std = rf_tree_preds.std(axis=0)
rf_lo  = 10 ** (rf_log_pred - rf_std)
rf_hi  = 10 ** (rf_log_pred + rf_std)

sigma_dt  = 10 ** dt_log_pred
sigma_rf  = 10 ** rf_log_pred
sigma_xgb = 10 ** xgb_log_pred

nn_model.eval()
with torch.no_grad():
    nn_log_pred = nn_model(
        torch.tensor(X_grid_nn_scaled, dtype=torch.float32).to(device)
    ).cpu().numpy()
sigma_nn = 10 ** nn_log_pred

# ENDF interpolated onto energy grid
lib_valid    = lib_df[lib_label] > 0
lib_energies = lib_df.loc[lib_valid, "Incident energy"].values
lib_xs       = lib_df.loc[lib_valid, lib_label].values

endf_interp = interp1d(
    np.log10(lib_energies), np.log10(lib_xs),
    kind="linear", bounds_error=False, fill_value=np.nan
)
sigma_endf = 10 ** endf_interp(logE_grid)

# Plot
fig, ax = paper_style.fig("wide")

# Highlight data gap region (Cobalt)
ax.axvspan(1, 1e3,
           alpha=0.08,
           color=paper_style.COLORS["sky"],
           label="No EXFOR data (1 eV\u20131 keV)",
           zorder=0)

# Mark ENDF first resonance (Cobalt)
ax.axvline(132.0,
           color="black",
           linewidth=0.8,
           linestyle="--",
           alpha=0.5,
           zorder=1,
           label="ENDF first resonance (132 eV)")

# Scatter EXFOR truth data
ax.scatter(
    df_true["Energy"], df_true["CrossSection"],
    s=4, alpha=0.3,
    color=paper_style.COLORS["sky"],
    zorder=2, label="EXFOR Data",
    rasterized=True,
)

# Plot predictions and ENDF
ax.plot(E_grid, sigma_endf, linewidth=1.8, linestyle="-",
        color="black",                       zorder=3, label=lib_label)
ax.plot(E_grid, sigma_dt,   linewidth=1.2, linestyle="--",
        color=paper_style.COLORS["orange"],  zorder=4, label="Decision Tree")
ax.plot(E_grid, sigma_rf,   linewidth=1.2, linestyle="-.",
        color=paper_style.COLORS["green"],   zorder=4, label="Random Forest")
ax.fill_between(E_grid, rf_lo, rf_hi,
                alpha=0.12,
                color=paper_style.COLORS["green"],
                zorder=3,
                label=r"RF $\pm1\sigma$")
ax.plot(E_grid, sigma_xgb,  linewidth=1.2, linestyle=":",
        color=paper_style.COLORS["purple"],  zorder=4, label="XGBoost")
ax.plot(E_grid, sigma_nn,   linewidth=1.8, linestyle=(0, (3, 1)),
        color=paper_style.COLORS["red"],     zorder=4, label="Neural Network")


# Format graph
ax.set_xscale("log")
ax.set_yscale("log")
#ax.set_xlim(E_min * 0.5, E_max * 2)
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"$\sigma(E)$ (barns)")
ax.legend(ncol=2)

plt.tight_layout()
paper_style.save(f"full_model_comparison_A{A_target}_MT{MT_target}.pdf")
plt.show()

# Per-reaction metrics on val set EXFOR points
df_eval = df_true[df_true.index.isin(df_val.index)].copy()
df_eval["logE"]   = np.log10(df_eval["Energy"])
df_eval["log_XS"] = np.log10(df_eval["CrossSection"])
y_eval            = df_eval["log_XS"].values

print(f"\nPer-reaction metrics (val set, N={len(df_eval):,}):")
print(f"Model/Library Log RMSE Log R²")
print("")

X_eval_tree = np.nan_to_num(df_eval[TREE_FEATURE_COLS].values, nan=0.0)
mask_tree   = np.isfinite(X_eval_tree).all(axis=1) & np.isfinite(y_eval)

X_eval_nn_scaled = scaler.transform(
    np.nan_to_num(df_eval[FEATURE_COLS].values, nan=0.0)
)
mask_nn = np.isfinite(X_eval_nn_scaled).all(axis=1) & np.isfinite(y_eval)

nn_model.eval()
with torch.no_grad():
    nn_eval_pred = nn_model(
        torch.tensor(X_eval_nn_scaled[mask_nn], dtype=torch.float32).to(device)
    ).cpu().numpy()

for name, preds, mask in [
    ("Decision Tree",  dt_model.predict(X_eval_tree[mask_tree]), mask_tree),
    ("Random Forest",  rf_model.predict(X_eval_tree[mask_tree]), mask_tree),
    ("XGBoost",        xgb_model.predict(X_eval_tree[mask_tree]), mask_tree),
    ("Neural Network", nn_eval_pred,                              mask_nn),
]:
    y = y_eval[mask]
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2   = r2_score(y, preds)
    print(f"{name} {rmse} {r2}")

in_range = (
    (df_eval["Energy"].values >= lib_energies.min()) &
    (df_eval["Energy"].values <= lib_energies.max())
)
lib_interp = endf_interp(df_eval.loc[in_range, "logE"].values)
finite     = np.isfinite(lib_interp)
y_lib      = y_eval[in_range][finite]
rmse = np.sqrt(mean_squared_error(y_lib, lib_interp[finite]))
r2   = r2_score(y_lib, lib_interp[finite])
print(f"{lib_label} {rmse} {r2}")

In [ ]:
# Co-59 Model resonance peak locations

# Restrict to resonance region only
mask_res = (E_grid > 100) & (E_grid < 400)
E_res    = E_grid[mask_res]

print(f"Type Model/Library Peak energy (eV) Peak σ (barns)")
print("")

# ENDF reference
print(f"Reference ENDF/B-VIII.0 132.0 849.5")
print("")

for name, sigma in [
    ("Decision Tree",  sigma_dt),
    ("Random Forest",  sigma_rf),
    ("XGBoost",        sigma_xgb),
    ("Neural Network", sigma_nn),
]:
    sigma_res = sigma[mask_res]
    peak_idx  = np.argmax(sigma_res)
    peak_E    = E_res[peak_idx]
    peak_xs   = sigma_res[peak_idx]
    print(f"ML Model {name} {peak_E} {peak_xs}")

In [ ]:
# Training Curves
train_losses = history["train_loss"]
val_losses   = history["val_loss"]
lr_schedule  = history["lr"]
lr_changes   = history["lr_updates"]

epochs     = list(range(1, len(train_losses) + 1))
best_epoch = int(np.argmin(val_losses)) + 1

fig, (ax1, ax2) = paper_style.fig("wide", nrows=2, sharex=True)

# Loss curves
ax1.plot(epochs, train_losses,
         color=paper_style.COLORS["blue"],
         linewidth=1.2, label="Train")
ax1.plot(epochs, val_losses,
         color=paper_style.COLORS["orange"],
         linewidth=1.2, label="Validation")
ax1.axvline(best_epoch,
            color=paper_style.COLORS["green"],
            linewidth=0.8, linestyle="--",
            label=f"Best: {min(val_losses):.4f} (epoch {best_epoch})")

for ep in lr_changes:
    ax1.axvline(ep,
                color=paper_style.COLORS["red"],
                linewidth=0.6, linestyle=":",
                alpha=0.5)
if lr_changes:
    ax1.axvline(lr_changes[0],
                color=paper_style.COLORS["red"],
                linewidth=0.6, linestyle=":",
                alpha=0.5, label="LR reduction")

ax1.set_ylabel("Loss")
ax1.legend()

# Learning rate schedule
ax2.plot(epochs, lr_schedule,
         color=paper_style.COLORS["red"],
         linewidth=1.2)
ax2.set_ylabel("Learning Rate")
ax2.set_xlabel("Epoch")
ax2.set_yscale("log")

plt.tight_layout()
paper_style.save("training_curves.pdf")
plt.show()

In [ ]:
# Save All Models
import os
import joblib

save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

# Neural Network
nn_save_path = os.path.join(save_dir, "nn_model.pt")
torch.save({
    "model_state_dict":     nn_model.state_dict(),
    "optimizer_state_dict": nn_optimizer.state_dict(),
    "best_val_loss":        nn_best_val_loss,
    "input_dim":            input_dim,
    "feature_cols":         FEATURE_COLS,
    "history":              history,
}, nn_save_path)
joblib.dump(scaler, os.path.join(save_dir, "nn_scaler.joblib"))

print(f"Neural Network to {nn_save_path}")
print(f"Input dim:        {input_dim}")
print(f"Best val loss:    {nn_best_val_loss:.4f}")
print(f"Epochs trained:   {len(history['train_loss'])}")

# Tree Models
tree_configs = [
    ("decision_tree", dt_model, dt_metrics),
    ("random_forest",  rf_model, rf_metrics),
    ("xgboost",        xgb_model, xgb_metrics),
]

for name, model, metrics in tree_configs:
    path = os.path.join(save_dir, f"{name}.joblib")
    joblib.dump({
        "model":        model,
        "feature_cols": FEATURE_COLS,
        "metrics":      metrics,
    }, path)
    size = os.path.getsize(path) / (1024 ** 2)
    print(f"{name} to {path}  ({size:.1f} MB)")

In [ ]:
# Load All Models
import torch
import joblib
import os

save_dir = "saved_models"
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Neural Network
checkpoint   = torch.load(os.path.join(save_dir, "nn_model.pt"),
                          map_location=device, weights_only=False)
FEATURE_COLS = checkpoint["feature_cols"]
input_dim    = checkpoint["input_dim"]
history      = checkpoint["history"]

nn_model = CrossSectionNet(input_dim).to(device)
nn_model.load_state_dict(checkpoint["model_state_dict"])
nn_model.eval()

scaler = joblib.load(os.path.join(save_dir, "nn_scaler.joblib"))

print(f"Neural Network loaded")
print(f"Input dim:      {input_dim}")
print(f"Best val loss:  {checkpoint['best_val_loss']:.4f}")
print(f"Epochs trained: {len(history['train_loss'])}")
print(f"Features:       {len(FEATURE_COLS)}")

# Tree Models
dt_bundle  = joblib.load(os.path.join(save_dir, "decision_tree.joblib"))
rf_bundle  = joblib.load(os.path.join(save_dir, "random_forest.joblib"))
xgb_bundle = joblib.load(os.path.join(save_dir, "xgboost.joblib"))

dt_model  = dt_bundle["model"]
rf_model  = rf_bundle["model"]
xgb_model = xgb_bundle["model"]
dt_metrics  = dt_bundle["metrics"]
rf_metrics  = rf_bundle["metrics"]
xgb_metrics = xgb_bundle["metrics"]

rf_importances = joblib.load(os.path.join(save_dir, "rf_importances.joblib"))

print(f"\nAll models loaded from '{save_dir}/'.")